
**1. Problem Statement:**

Customer support tickets are written in free text, making it difficult
to automatically categorize them.

Objective:

Use a Large Language Model (LLM) to automatically tag support tickets
into categories using:
- Zero-shot learning
- Few-shot learning (prompt engineering)

The system will output top 3 most relevant tags for each ticket.

In [1]:
!pip install -q transformers torch pandas scikit-learn matplotlib

In [2]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

**2. Dataset Loading and Preprocessing**

In [3]:
df = pd.read_csv("customer_support_tickets.csv")

df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [4]:
text_column = "Ticket Description"     # or "description"
label_column = "Ticket Type"       # or "issue_type"

df = df[[text_column, label_column]].dropna()

df.rename(columns={
    text_column: "ticket",
    label_column: "label"
}, inplace=True)

df.head()

,ticket,label
0,I'm having an issue with the {product_purchase...,Technical issue
1,I'm having an issue with the {product_purchase...,Technical issue
2,I'm facing a problem with my {product_purchase...,Technical issue
3,I'm having an issue with the {product_purchase...,Billing inquiry
4,I'm having an issue with the {product_purchase...,Billing inquiry


In [5]:
labels = df["label"].unique().tolist()
labels

['Technical issue',
 'Billing inquiry',
 'Cancellation request',
 'Product inquiry',
 'Refund request']

In [6]:
#Load LLM Model (Zero-shot)
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
#Zero-Shot Prediction
def zero_shot_predict(text):
    result = classifier(text, labels)
    return result["labels"][:3]   # Top 3 tags

In [ ]:
def batch_process(texts, batch_size=16):
    all_results = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = classifier(batch, labels)
        all_results.extend([res["labels"][:3] for res in results])

    return all_results

df["zero_shot_tags"] = batch_process(df["ticket"].tolist(), batch_size=16)

In [ ]:
#Few-Shot Prompt Engineering
def few_shot_prompt(text):
    prompt = f"""
You are a support ticket classifier.

Examples:
Text: My internet is not working
Tags: Technical Issue

Text: I was charged twice
Tags: Billing

Text: I cannot login to my account
Tags: Account

Now classify:
Text: {text}
Tags:
"""
    return prompt

In [ ]:
#Few-shot prediction
def few_shot_predict(text):
    result = classifier(few_shot_prompt(text), labels)
    return result["labels"][:3]

In [ ]:
df["few_shot_tags"] = df["ticket"].apply(few_shot_predict)

**3. Evaluation**

In [ ]:
df["zero_top1"] = df["zero_shot_tags"].apply(lambda x: x[0])
df["few_top1"] = df["few_shot_tags"].apply(lambda x: x[0])

zero_acc = accuracy_score(df["label"], df["zero_top1"])
few_acc = accuracy_score(df["label"], df["few_top1"])

print("Zero-shot Accuracy:", zero_acc)
print("Few-shot Accuracy:", few_acc)

**4. Visualization**

In [ ]:
models = ["Zero-shot", "Few-shot"]
scores = [zero_acc, few_acc]

plt.bar(models, scores)
plt.title("Zero-shot vs Few-shot Accuracy")
plt.ylabel("Accuracy")
plt.show()

**5. Results & Insights:**

- Zero-shot classification works without training but has moderate accuracy.
- Few-shot prompting improves performance using examples.
- LLM successfully predicts top 3 relevant tags.
- Prompt engineering plays a key role in improving results.

**Conclusion:**

LLMs can effectively classify support tickets without traditional
training. Few-shot learning significantly improves performance and
is useful for real-world applications.